In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.db import supabase, fetch_all
from src.features import build_run_features
from src.scoring import compute_scores, ml_enrichment, print_run_report
from config.scoring_config import SCORING_CONFIG


In [ ]:
print("📥 Chargement des données...")

df_runs      = fetch_all("Run")
df_stages    = fetch_all("Stage")
df_rooms     = fetch_all("Room")
df_pstates   = fetch_all("PlayerState")
df_inventory = fetch_all("Inventory")
df_passive   = fetch_all("PassiveItem")
df_active    = fetch_all("ActiveItem")
df_trinkets  = fetch_all("Trinket")
df_pactive   = fetch_all("PlayerStateActive")
df_ptrinket  = fetch_all("PlayerStateTrinket")
df_rmonster  = fetch_all("RoomMonster")
df_rboss     = fetch_all("RoomBoss")

print(f"✅ {len(df_runs)} runs chargées")


In [ ]:
print("⚙️  Construction des features...")

df_features = build_run_features(
    df_runs, df_stages, df_rooms, df_pstates,
    df_inventory, df_passive, df_pactive,
    df_ptrinket, df_rmonster, df_rboss
)

print(f"✅ {df_features.shape[1] - 1} features pour {len(df_features)} runs")
df_features.head()


In [ ]:
print("🏆 Calcul des scores...")
df_scored = compute_scores(df_features, SCORING_CONFIG)

cols = ["run_id", "score", "grade", "rank", "victory",
        "nb_stages", "dps_proxy", "total_damage_taken",
        "nb_passive_items", "avg_passive_quality"]
cols = [c for c in cols if c in df_scored.columns]

df_scored[cols].head(10)


In [ ]:
df_scored = ml_enrichment(df_scored, SCORING_CONFIG)
df_scored[["run_id", "score", "grade", "cluster_label", "anomaly_label"]].head(10)


In [ ]:
# Remplace l'id par celui que tu veux inspecter
print_run_report(df_scored, run_id=df_scored.iloc[0]["run_id"])


In [ ]:
df_scored.to_csv("../data/run_scores.csv", index=False)
print("💾 Export terminé → data/run_scores.csv")
